# 6. Embedding 与向量存储（精简）

来源：《尚硅谷-10-RAG.pdf》，第 46–52 页，截止 2.5.2；对应课件 **2.4、2.5.1–2.5.2**。页码均指 PDF 阅读器中的页序。

整理方式：原理保留关键逻辑；第 2 章示例精简，重点学习拆分器源码与递归拆分。
只记录到 2.5.2，不收录 2.5.3 案例。

**运行约定**：在 `D:/grow/langchain/7_RAG` 打开本文件，选择
`D:/conda/envs/langchain/python.exe` 对应的 Python 内核；代码单元按本文件顺序运行。
各 Notebook 的变量相互独立。使用 `len` 的示例以 Python 字符长度计数，不能据此推断 Token 数。

## 6.1 Embedding：把文本变成可比较的数值

Embedding 模型把文本编码成定长向量。语义相近的文本，在相应模型的向量空间里通常更接近。
向量的每一维一般不能直接命名为「时间、地点、相机型号」，课件照片例子只是空间关系的类比。

| 接口 | 输入 | 输出 | 常见位置 |
| --- | --- | --- | --- |
| `embed_query(text)` | `str` | `list[float]` | 用户问题向量化 |
| `embed_documents(texts)` | `list[str]` | `list[list[float]]` | 文本块批量向量化并建索引 |

`embed_documents()` 的参数是文本列表，不是直接传 `list[Document]`。
已有 Document 时，用 `[doc.page_content for doc in docs]` 提取正文；metadata 另行保留。

文档和查询应使用**同一模型，或服务方明确设计为匹配的编码器**，并保持维度和预处理约定相容。
维度相同不代表向量空间相同；更换嵌入模型后通常需要重新生成文档向量和索引。

## 6.2 模型选择与初始化：不复制平台套餐

课件列举 BGE 系列、`bge-m3`、`text-embedding-3-small`、`text-embedding-3-large`，
并演示 `init_embeddings()` 与 `OpenAIEmbeddings` 的初始化方式。
这里保留 API 的学习重点，不把课件中转站、历史免费额度、价格或固定窗口当作当前承诺。

选型主要看语言与领域、文本长度上限、检索效果、延迟、费用，以及本地 / API 部署方式。
维度越高不等于检索效果一定越好，也会带来存储与计算成本。

参数关系：`model` 必须是对应服务商支持的真实 ID；`base_url` 是该服务商的入口；
`api_key` 必须与这个服务匹配。支持 OpenAI 协议也不代表所有参数和 Tokenizer 约定都完全一样。

## 6.3 保留最短接口示例

下列为 **Markdown 中的可复制代码**，需要你已初始化的真实 `embedding_model`。
本次未执行，也没有提供模拟向量冒充模型输出。

```python
# embedding_model：按所选服务的真实模型 ID、base_url、api_key 初始化。
text = "What was the name mentioned in the conversation?"
embedded_query = embedding_model.embed_query(text)
print(embedded_query[:5])
print("向量维度:", len(embedded_query))

texts = ["Hi there!", "Oh, hello!", "What's your name?"]
embeded_docs = embedding_model.embed_documents(texts)
print("向量数量:", len(embeded_docs))
print("第一条向量维度:", len(embeded_docs[0]))
```

从 Document 到文本的连接只需：

```python
texts = [doc.page_content for doc in docs]
embeded_docs = embedding_model.embed_documents(texts)
# embeded_docs[i] 对应 texts[i]；doc.metadata 仍需与该文本关联保存。
```

`embedded_query` / `embeded_docs` 沿用课件命名。切换模型后，不能沿用课件展示的数值和维度作固定断言。

## 6.4 Vector Store：保存并检索文本向量

一条索引记录通常关联：**ID、向量、原始文本块、metadata**。
检索时把问题向量与文档向量按选定度量比较，取相关候选，再返回对应文档。
相似性检索是近邻匹配，结果不等于关键词精确匹配，也不保证就是事实正确的答案。

余弦相似度可以写为：

$$\operatorname{cosine}(a,b)=\frac{a\cdot b}{\lVert a\rVert\lVert b\rVert}$$

向量数据库还可能使用内积、欧氏距离等度量和近似近邻索引。
相似度 / 距离分数的方向与范围取决于实现，不能一律认为「越大越相关」。
建立索引也不意味着每次都顺序遍历所有向量。

## 6.5 课件列举的常用组件

| 组件 | 只记定位 |
| --- | --- |
| FAISS | 高效向量相似性搜索与聚类的库，不宜简单等同于完整数据库服务 |
| Chroma | 常见的轻量向量存储选择 |
| Milvus | 面向向量检索的数据库，课件后面的案例使用它 |
| pgvector | PostgreSQL 的向量类型与相似性搜索扩展 |
| Redis / Elasticsearch | 课件列举的可支持向量检索的系统 |
| Pinecone | 托管向量数据库服务 |

此处只做课件范围内的定位比较，不作当前产品许可、套餐或性能承诺。
不展开部署、连接和案例工程。

## 6.6 Vector Store 与 Retriever 的关系

Vector Store 提供向量存取与相似性搜索；Retriever 对外暴露「给定查询，返回相关 Document」的接口。
向量库可以包装为检索器，但检索器也可以来自其他搜索方式。
检索完成后，还需要把正文和来源组织进 Prompt，让模型生成有资料依据的回答。

**本章记忆链**：加载为 Document → 切分成 Chunk → 对正文 Embedding → 关联 metadata 存储
→ 查询向量化 → 检索文档 → 生成回答。

## 笔记导航

1. [7_1_RAG原理与流程](7_1_RAG原理与流程.ipynb)
2. [7_2_Document与文档加载](7_2_Document与文档加载.ipynb)
3. [7_3_TextSplitter源码与Character切分](7_3_TextSplitter源码与Character切分.ipynb)
4. [7_4_RecursiveCharacterTextSplitter详解](7_4_RecursiveCharacterTextSplitter详解.ipynb)
5. [7_5_其他拆分方式速记](7_5_其他拆分方式速记.ipynb)
6. [7_6_Embedding与向量存储](7_6_Embedding与向量存储.ipynb)